# 🧪 Test Saved Indian Language Classifier Model

Upload your saved `.keras` model and test it on any audio file to predict the language.

## Step 1: Install Dependencies

In [ ]:
!pip install -q librosa pydub

## Step 2: Upload Your Saved Model & Label Mapping

Upload these files from the downloaded `saved_model.zip`:
1. `indian_language_classifier.keras`
2. `label_mapping.json`

In [ ]:
from google.colab import files
import os

print('📤 Upload your saved model (.keras) and label_mapping.json files:')
uploaded = files.upload()

for fname in uploaded:
    print(f'  ✅ Uploaded: {fname} ({len(uploaded[fname]):,} bytes)')

## Step 3: Load the Model

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Find the model file (supports .keras, .h5, or directory)
model_file = None
for f in os.listdir('.'):
    if f.endswith('.keras') or f.endswith('.h5'):
        model_file = f
        break

if model_file is None:
    print('❌ No model file found! Please upload .keras or .h5 file.')
else:
    model = load_model(model_file)
    print(f'✅ Model loaded from: {model_file}')
    print(f'   Input shape: {model.input_shape}')
    print(f'   Output classes: {model.output_shape[1]}')

# Load label mapping
with open('label_mapping.json', 'r') as f:
    label_data = json.load(f)

idx_to_label = label_data['index_to_label']
print(f'\n🌐 Languages: {", ".join(idx_to_label.values())}')

## Step 4: Upload Audio File to Test

In [ ]:
print('🎵 Upload an audio file (.mp3 or .wav) to test:')
audio_uploaded = files.upload()
test_audio_path = list(audio_uploaded.keys())[0]
print(f'  ✅ Audio file: {test_audio_path}')

## Step 5: Predict the Language

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
from pydub import AudioSegment
import IPython.display as ipd
import io
import warnings
warnings.filterwarnings('ignore')

def extract_mfcc(file_path, sr=22050, n_mfcc=40, max_len=216):
    """Extract MFCC features from an audio file."""
    try:
        y, sr = librosa.load(file_path, sr=sr, duration=5)
    except:
        audio = AudioSegment.from_mp3(file_path)
        buffer = io.BytesIO()
        audio.export(buffer, format='wav')
        buffer.seek(0)
        y, sr = librosa.load(buffer, sr=sr, duration=5)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

    if mfccs.shape[1] < max_len:
        pad_width = max_len - mfccs.shape[1]
        mfccs = np.pad(mfccs, ((0, 0), (0, pad_width)), mode='constant')
    else:
        mfccs = mfccs[:, :max_len]

    return mfccs, y, sr


# Extract features
mfcc_features, audio_signal, sample_rate = extract_mfcc(test_audio_path)

# Reshape: (1, timesteps, n_mfcc)
X_input = mfcc_features.T[np.newaxis, :, :]

# Predict
prediction = model.predict(X_input, verbose=0)
predicted_class = np.argmax(prediction)
confidence = prediction[0][predicted_class] * 100
predicted_language = idx_to_label[str(predicted_class)]

# Display results
print('=' * 50)
print('🎯 PREDICTION RESULT')
print('=' * 50)
print(f'   Audio File:     {test_audio_path}')
print(f'   Predicted:      {predicted_language}')
print(f'   Confidence:     {confidence:.2f}%')
print('=' * 50)

# Play audio
print('\n🔊 Audio Playback:')
ipd.display(ipd.Audio(audio_signal, rate=sample_rate))

# Show probabilities for all languages
print('\n📊 Probabilities for All Languages:')
lang_names = [idx_to_label[str(i)] for i in range(len(idx_to_label))]
probs = prediction[0] * 100

plt.figure(figsize=(10, 5))
colors = ['#2ecc71' if i == predicted_class else '#3498db' for i in range(len(lang_names))]
bars = plt.barh(lang_names, probs, color=colors, edgecolor='black', linewidth=0.5)
plt.xlabel('Probability (%)', fontsize=12)
plt.title(f'🎙️ Predicted: {predicted_language} ({confidence:.1f}%)', fontsize=14, fontweight='bold')

for bar, prob in zip(bars, probs):
    if prob > 2:
        plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2.,
                 f'{prob:.1f}%', va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

## Step 6: Test Another Audio File (Optional)

Re-run the cells in **Step 4** and **Step 5** to test with a different audio file.